# Amparo -- Baseline y fine-tuning LoRA del agente legal

Este notebook:
1. Carga `dataset_legal.jsonl` (**1536 ejemplos en 33 categorias**) y separa una validacion estratificada por categoria que **nunca** se usa para entrenar.
2. Corre el modelo base sin modificar sobre esa validacion (**baseline**).
3. Entrena un adaptador **LoRA** sobre el resto del dataset.
4. Vuelve a correr el modelo ya afinado sobre la misma validacion.
5. Compara baseline vs. afinado -- esa comparacion es la evidencia real de que el fine-tuning mejoro algo, no solo una suposicion.

**Sobre esta version del dataset.** La version anterior (1320 ejemplos) daba consejo generico sin decirle a la persona a donde acudir: solo el 14% de las respuestas nombraba un mecanismo legal concreto, lo que contradecia el principio 3 de `PRODUCT.md`. El dataset se reescribio para que cada respuesta señale la ruta real (tutela, Inspeccion del Trabajo, Comisaria de Familia, centro de conciliacion, etc.) y la evidencia a reunir; hoy lo cumple el **99.7%**. Tambien se agregaron tres categorias que faltaban y concentraban los casos de urgencia: servicios publicos domiciliarios, violencia intrafamiliar y penal basico/victimas.

Lo que el dataset **no** enseña, a proposito: numeros de articulo, de ley ni plazos exactos. Ese conocimiento verificable lo aporta el sistema RAG de M3 contra el corpus normativo real; si el modelo lo memorizara, inventaria precision que no puede respaldar.

Antes de correr: `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU` (una T4 gratis alcanza usando 4-bit/QLoRA).

**Nota sobre las librerias:** `transformers`, `peft` y `trl` cambian su API con cierta frecuencia. Este notebook esta escrito contra las versiones estables mas recientes conocidas al momento de escribirlo; si algun cambio de firma da error, revisa el changelog de la libreria correspondiente -- la logica de cada celda (que hace y por que) sigue siendo valida aunque cambie algun nombre de parametro.

In [1]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets wandb
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires 

## Configuracion

Cambia `MODEL_ID` para comparar Qwen2.5 7B vs. Llama 3.1 8B (o el que haya ganado en el comparador local `tools/model_comparator`).

In [2]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # alternativa: "meta-llama/Llama-3.1-8B-Instruct"

VAL_FRACTION = 0.15   # % por categoria reservado para validacion (nunca se entrena con esto)
RANDOM_SEED = 42

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8   # batch efectivo = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS
MAX_SEQ_LENGTH = 1024

# Era 300: a ese techo el baseline (que responde largo, con listas) quedaba
# cortado a mitad de frase en la mayoria de los casos, y la evaluacion lo
# castigaba por incompleto en vez de por incorrecto. 900 es el mismo valor
# que MAX_NEW_TOKENS_GENERATION de tools/evaluation/config.py, para que la
# cifra de M1 y la de M2 se midan bajo las mismas condiciones.
MAX_NEW_TOKENS_EVAL = 900
OUTPUT_DIR = "/content/amparo-lora"

## Monta tu Google Drive y ubica el dataset

Sube `dataset_legal.jsonl` a `MyDrive/Colab Notebooks/Amparo/dataset_legal.jsonl` (ajusta `DATASET_PATH` si usaste otra ruta). Con Drive no tienes que volver a subirlo cada vez que el entorno de Colab se reinicia.

**Importante:** sube la version actual del repositorio (1536 ejemplos). Si en Drive quedo una version anterior (1320 o 1410), el entrenamiento correria con el dataset viejo sin dar ningun error visible. La celda siguiente imprime el total: verifica que diga 1536 antes de seguir.

El dataset no se edita a mano: se construye desde las fuentes versionadas en `data/dataset_src/` con `python -m tools.dataset_build`, que solo lo escribe si pasa las ocho puertas de calidad de `tools/dataset_quality.py`. Asi queda trazable con que datos se entreno y cualquier cambio es revisable ejemplo por ejemplo en un diff.

In [3]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Colab Notebooks/Amparo'
SALIDA_DIR = f'{DRIVE_DIR}/evaluacion'   # resultados de la corrida, no en /content
DATASET_PATH = f'{DRIVE_DIR}/dataset_legal.jsonl'
ADAPTER_DIR = f'{DRIVE_DIR}/amparo-lora-adapter'

import os
os.makedirs(SALIDA_DIR, exist_ok=True)
print(f'Dataset: {DATASET_PATH}')
print(f'Resultados -> {SALIDA_DIR}')

Mounted at /content/drive
Dataset: /content/drive/MyDrive/Colab Notebooks/Amparo/dataset_legal.jsonl
Resultados -> /content/drive/MyDrive/Colab Notebooks/Amparo/evaluacion


## Cargar y separar train / validacion (estratificado por categoria)

In [4]:
import json
import random
from collections import defaultdict

random.seed(RANDOM_SEED)

records = [json.loads(line) for line in open(DATASET_PATH, encoding='utf-8')]

by_category = defaultdict(list)
for r in records:
    by_category[r['category']].append(r)

train_records, val_records = [], []
for category, items in by_category.items():
    items = items[:]
    random.shuffle(items)
    n_val = max(1, round(len(items) * VAL_FRACTION))
    val_records.extend(items[:n_val])
    train_records.extend(items[n_val:])

random.shuffle(train_records)
random.shuffle(val_records)

print(f'Total: {len(records)} | Train: {len(train_records)} | Validacion: {len(val_records)}')
# Guarda contra el fallo silencioso de entrenar con una copia vieja del dataset
# que haya quedado en Drive: sin esto, el notebook corre igual y nadie se entera.
assert len(records) == 1536, (
    f'El dataset tiene {len(records)} ejemplos y se esperaban 1536. '
    'Probablemente quedo en Drive una version anterior: subela de nuevo desde el repo.'
)
for category in sorted(by_category):
    n_val = sum(1 for r in val_records if r['category'] == category)
    n_train = sum(1 for r in train_records if r['category'] == category)
    print(f'  {category:45s} train={n_train:3d}  val={n_val:3d}')

Total: 1536 | Train: 1305 | Validacion: 231
  Acceso a informacion publica                  train= 43  val=  8
  Accidentes de transito                        train= 52  val=  9
  Arriendo                                      train= 52  val=  9
  Comparendos de transito                       train= 52  val=  9
  Conciliacion prejudicial                      train= 43  val=  8
  Contratacion estatal y facturacion            train= 43  val=  8
  Contratos empresariales (B2B)                 train= 43  val=  8
  Derecho administrativo general                train= 43  val=  8
  Derecho ambiental sancionatorio               train= 43  val=  8
  Derecho contractual general                   train= 43  val=  8
  Derecho de familia - alimentos                train= 43  val=  8
  Despido                                       train= 53  val=  9
  Educacion / debido proceso disciplinario      train= 43  val=  8
  Embargos                                      train= 52  val=  9
  Fuera del derech

## Cargar el modelo base en 4-bit (QLoRA)

Se carga cuantizado en 4-bit para que quepa comodamente en una GPU gratuita de Colab (T4, ~16GB).

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

## Metrica de similitud

Misma heuristica lexica (difflib) que usa el comparador local en `tools/model_comparator/metrics.py`, para que los resultados sean comparables entre ambas herramientas. No es una metrica juridica rigurosa de correccion legal, solo sirve para ordenar/comparar respuestas de forma consistente.

**Por que esta cifra puede no coincidir con la del scorecard de M2.** Aqui se mide el modelo que acaba de quedar en memoria despues de entrenar; M2 recarga el adaptador guardado en Drive sobre el modelo base, y suele correr en otra GPU. Con decodificacion greedy, cualquier diferencia numerica hace que el texto generado diverja desde el primer token ambiguo, asi que las dos mediciones son legitimas pero no identicas. La cifra de referencia del proyecto es la del scorecard de M2, porque es la que queda respaldada por un artefacto comiteado con su manifiesto (commit, semilla, hardware).

In [6]:
import re
from difflib import SequenceMatcher

SYSTEM_PROMPT = records[0]['messages'][0]['content']

# --- Metrica lexica (secundaria) -------------------------------------------
# Compara el fraseo contra la respuesta de referencia. NO dice si la respuesta
# es juridicamente util: dos respuestas igual de correctas redactadas distinto
# puntuan bajo. Se reporta como dato de apoyo, no como la cifra principal.
def similarity_pct(expected: str, actual: str) -> float:
    if not expected or not actual:
        return 0.0
    ratio = SequenceMatcher(None, expected.strip().lower(), actual.strip().lower()).ratio()
    return round(ratio * 100, 1)


# --- Metricas de dominio (principales) -------------------------------------
# Miden lo que el producto promete, y son verificables sin juez LLM.
#
# 1) Nombrar la ruta legal concreta es el principio 3 de PRODUCT.md: la respuesta
#    debe apuntar a una figura juridica real (tutela, Inspeccion del Trabajo,
#    Comisaria de Familia...) y no quedarse en consejo generico.
# 2) Citar normas que no se pueden verificar es el fallo que el producto no puede
#    permitirse. El dataset de M1 no enseña numeros de articulo a proposito -- ese
#    conocimiento lo aporta el RAG de M3 contra el corpus real -- asi que toda
#    cita que aparezca aqui sale de la memoria del modelo. Deberia ser 0%.
# El catalogo de rutas se genera desde tools/dataset_quality.py (el mismo con
# el que se valido el dataset) y lo vigila tests/test_notebook_m1_coherencia.py.
# No editarlo a mano: una version recortada produce falsos negativos y
# subestima al modelo sin que nada falle. Ya paso una vez: una respuesta que
# decia "apelar ante la Secretaria de Transito" se contaba como "no nombra
# ruta" porque el patron solo tenia "apelacion".
MECANISMO = re.compile(
    r"""(autoridad|[Ss]ecretar[ií]a|organismo|entidad) de tr[aá]nsito|\bSIMIT\b|inspector de tr[aá]nsito|impugna(r|cion|ción)|descargos|comparendo ante|[Aa]utoridad ambiental|\bCAR\b|\bANLA\b|[Cc]orporaci[oó]n [Aa]ut[oó]noma|[Ss]ecretar[ií]a (Distrital |Municipal )?de [Aa]mbiente|[Cc][aá]mara de [Cc]omercio|[Nn]otar[ií]a|[Oo]ficina de [Ii]nstrumentos [Pp][uú]blicos|\bORIP\b|\bIGAC\b|[Cc]olpensiones|fondo de pensiones|administradora de pensiones|[Cc]aja de [Cc]ompensaci[oó]n|operador de aportes|[Cc]urador [Uu]rbano|[Pp]laneaci[oó]n [Mm]unicipal|[Ss]ecretar[ií]a de [Pp]laneaci[oó]n|[Aa]lcald[ií]a|[Dd]efensor[ií]a del [Pp]ueblo|[Pp]ersoner[ií]a|[Pp]rocuradur[ií]a|[Cc]ontralor[ií]a|[Vv]eedur[ií]a|[Ii]nspecci[oó]n del? [Tt]rabajo|[Ii]nspector del? [Tt]rabajo|[Mm]inisterio del? [Tt]rabajo|[Ll][ií]nea 123|\b123\b|[Ll][ií]nea 155|[Pp]olic[ií]a|urgencias|[Mm]igraci[oó]n [Cc]olombia|[Rr]egistradur[ií]a|\bRUNT\b|[Ss]ecretar[ií]a de [Ee]ducaci[oó]n|[Mm]inisterio de [Ee]ducaci[oó]n|manual de convivencia|reglamento (acad[eé]mico|estudiantil)|[Ss]uperintendencia|\bSIC\b|\bSuperSalud\b|\bSupersalud\b|\bARL\b|\bEPS\b y la \bARL\b|[Jj]unta de [Cc]alificaci[oó]n|\bDIAN\b|\bUGPP\b|\bICBF\b|[Bb]ienestar [Ff]amiliar|[Dd]efensor de [Ff]amilia|[Cc]omisar[ií]a|r[eé]gimen de visitas|\bcustodia\b|alimentos provisionales|\bsucesi[oó]n\b|[Cc]onsulado|[Cc]anciller[ií]a|\bPQRS?\b|reclamaci[oó]n formal|derecho de reclamaci[oó]n|reclamaci[oó]n directa|\bSECOP\b|supervisor del contrato|ordenador del gasto|agencia de contrataci[oó]n|\bSENA\b|\bSOAT\b|aseguradora|[Mm]edicina [Ll]egal|\bdesacato\b|\bnulidad\b|revocatoria directa|\bquerellas?\b|\breintegro\b|fuero (de maternidad|sindical|de salud|circunstancial)|estabilidad laboral reforzada|permiso del inspector|\breposici[oó]n\b|\bapelaci[oó]n\b|recursos? de (queja|s[uú]plica)|recursos? de ley|v[ií]a gubernativa|insistencia|\btutelas?\b|acci[oó]n de tutela|acci[oó]n (popular|de grupo)|acci[oó]n de cumplimiento|central(es)? de riesgo|[Dd]atacr[eé]dito|[Tt]ransUni[oó]n|[Dd]efensor del [Cc]onsumidor [Ff]inanciero|casa de cobranza|paz y salvo|comit[eé] de (convivencia|seguridad y salud)|\bCOPASST\b|conciliaci[oó]n|conciliar ante|centro de conciliaci[oó]n|consultorio jur[ií]dico|\babogad[oa]s?\b|contencioso administrativ|nulidad y restablecimiento|control urbano|reconocimiento de (la )?edificaci[oó]n|espacio p[uú]blico|[Mm]inisterio de [Cc]ultura|demand(a|ar|as)\b|juez (laboral|civil|de familia|administrativo|de peque[nñ]as causas)|juzgado|v[ií]a judicial|proceso (ordinario|verbal|declarativo)|denuncia(r|s)? (penal|ante la [Ff]iscal)|\bFiscal[ií]a\b|\bURI\b|\bSAU\b|[Cc]asa de [Jj]usticia|derecho de petici[oó]n|h[aá]beas data|insolvencia de persona natural|insolvencia econ[oó]mica|medida de protecci[oó]n|[Cc]omisar[ií]a de [Ff]amilia|objeta(r|la|lo|ndo)?\b|oposici[oó]n a la (diligencia|medida)|levantamiento (del|de la) (embargo|medida)|excepciones de m[eé]rito|proceso ejecutivo|cobro ejecutivo|t[ií]tulo ejecutivo|protecci[oó]n al consumidor|registro de marca|infracci[oó]n marcaria|competencia desleal|oposici[oó]n al registro|cancelaci[oó]n por no uso|b[uú]squeda de antecedentes|[Dd]erecho de [Aa]utor|\bInvima\b|requerimiento escrito|requiere por escrito|(resoluci[oó]n|terminaci[oó]n) del contrato|junta o asamblea de socios|restituci[oó]n (del? )?(inmueble|bien)|proceso de restituci[oó]n|lanzamiento por ocupaci[oó]n|silencio administrativo|tribunal de arbitramento|cl[aá]usula compromisoria|centro de arbitraje|arbitraje""",
    re.I)

# La definicion de "cita que el modelo no puede verificar" es la MISMA que usa
# el harness de M2 (tools/evaluation/domain_metric.py). Antes cada milestone
# tenia la suya y reportaban cifras no comparables de la misma propiedad.
# tests/test_notebook_m1_coherencia.py falla si esta copia se desincroniza.
CITA_NO_VERIFICABLE = re.compile(
    r"""\bLey\s+\d+|\bDecreto\s+\d+|\bArt(?:i|í)culo\s+\d+|\bArt\.\s*\d+|\bSentencia\s+(?:T|C|SU)[\s\-]?\d+|\bResoluci(?:o|ó)n\s+\d+""",
    re.I)


def nombra_mecanismo(texto: str) -> bool:
    return bool(MECANISMO.search(texto or ''))


def cita_no_verificable(texto: str) -> bool:
    return bool(CITA_NO_VERIFICABLE.search(texto or ''))


@torch.no_grad()
def generate_response(model, query: str) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': query},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS_EVAL,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def evaluate(model, val_set, label: str):
    # Imprime progreso en CADA ejemplo con ETA: son ~231 generaciones de varios
    # segundos cada una, y sin señal constante la celda parece congelada.
    import time
    results, t0 = [], time.perf_counter()
    for i, item in enumerate(val_set, start=1):
        query = item['messages'][1]['content']
        expected = item['messages'][2]['content']
        actual = generate_response(model, query)
        results.append({
            **item,
            'generated': actual,
            'similarity': similarity_pct(expected, actual),
            'nombra_mecanismo': nombra_mecanismo(actual),
            'cita_no_verificable': cita_no_verificable(actual),
        })
        transcurrido = time.perf_counter() - t0
        eta = transcurrido / i * (len(val_set) - i) / 60
        print(f"[{label}] {i}/{len(val_set)}  "
              f"ruta={'si' if results[-1]['nombra_mecanismo'] else 'no '}  "
              f"{transcurrido / i:.1f}s/ej  ETA ~{eta:.1f} min", flush=True)
    n = len(results)
    print(f"\n[{label}] ruta legal: {sum(r['nombra_mecanismo'] for r in results) / n:.1%}"
          f" | citas no verificables: {sum(r['cita_no_verificable'] for r in results) / n:.1%}"
          f" | similitud lexica: {sum(r['similarity'] for r in results) / n:.1f}")
    return results


def guardar_resultados(nombre, datos):
    ruta = f'{SALIDA_DIR}/{nombre}'
    with open(ruta, 'w', encoding='utf-8') as f:
        for r in datos:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'Guardado en Drive: evaluacion/{nombre} ({len(datos)} registros)')

## Paso 1 -- Baseline (modelo base, sin fine-tuning)

Esto puede tardar varios minutos segun el tamano de la validacion.

In [7]:
baseline_results = evaluate(model, val_records, label='baseline')
guardar_resultados('baseline_results.jsonl', baseline_results)

[baseline] 1/231  ruta=si  22.5s/ej  ETA ~86.4 min
[baseline] 2/231  ruta=si  19.1s/ej  ETA ~72.9 min
[baseline] 3/231  ruta=si  23.4s/ej  ETA ~88.8 min
[baseline] 4/231  ruta=si  22.4s/ej  ETA ~84.9 min
[baseline] 5/231  ruta=si  22.0s/ej  ETA ~83.0 min
[baseline] 6/231  ruta=si  21.9s/ej  ETA ~82.2 min
[baseline] 7/231  ruta=si  21.9s/ej  ETA ~81.9 min
[baseline] 8/231  ruta=si  21.8s/ej  ETA ~80.9 min
[baseline] 9/231  ruta=si  21.0s/ej  ETA ~77.7 min
[baseline] 10/231  ruta=si  20.9s/ej  ETA ~76.9 min
[baseline] 11/231  ruta=si  20.6s/ej  ETA ~75.7 min
[baseline] 12/231  ruta=si  21.1s/ej  ETA ~77.1 min
[baseline] 13/231  ruta=si  21.3s/ej  ETA ~77.4 min
[baseline] 14/231  ruta=si  20.4s/ej  ETA ~73.6 min
[baseline] 15/231  ruta=si  20.9s/ej  ETA ~75.2 min
[baseline] 16/231  ruta=si  20.7s/ej  ETA ~74.2 min
[baseline] 17/231  ruta=si  20.7s/ej  ETA ~74.0 min
[baseline] 18/231  ruta=si  20.7s/ej  ETA ~73.4 min
[baseline] 19/231  ruta=si  20.6s/ej  ETA ~72.9 min
[baseline] 20/231  ru

## Paso 2 -- Fine-tuning con LoRA

### Antes de entrenar: conecta Weights & Biases

M1 pide dejar registrada la curva de perdida del entrenamiento como evidencia visual de que el modelo esta aprendiendo. Crea una cuenta gratuita en wandb.ai si no tienes una, y ten a mano tu API key (la consigues en https://wandb.ai/authorize).

**Si corres este notebook desde VSCode** (extension de Colab) en vez del navegador: `wandb.login()` te pide la key en un recuadro de entrada; si se queda esperando sin mostrarlo, usa `wandb.login(key="tu_key")` en su lugar. Este notebook no depende de los Secrets de Colab (`userdata.get`), que es lo unico que no funciona fuera del navegador.

In [8]:
import wandb

wandb.login()  # pega tu API key cuando te la pida (solo la primera vez)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tomasposada67 (tomasposada67-universidad-eafit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


In [10]:
from datasets import Dataset


def format_for_training(record):
    return {'text': tokenizer.apply_chat_template(record['messages'], tokenize=False)}


train_dataset = Dataset.from_list(train_records).map(format_for_training)

Map:   0%|          | 0/1305 [00:00<?, ? examples/s]

In [11]:
wandb.init(
    project='amparo-legal-finetune',
    name=f"lora-{MODEL_ID.split('/')[-1]}",
    config={
        'model_id': MODEL_ID,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
    },
)

from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_strategy='epoch',
    bf16=True,
    dataset_text_field='text',
    max_length=MAX_SEQ_LENGTH,
    report_to='wandb',
    run_name=f"lora-{MODEL_ID.split('/')[-1]}",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()

print('Dashboard de W&B (guarda este link para tu informe de M1):', wandb.run.url)
wandb.finish()

Tokenizing train dataset:   0%|          | 0/1305 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1305 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1305 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1305 [00:00<?, ? examples/s]

Step,Training Loss
10,2.255746
20,1.049836
30,0.773442
40,0.737224
50,0.702669
60,0.686011
70,0.674688
80,0.661286
90,0.634739
100,0.631928


Dashboard de W&B (guarda este link para tu informe de M1): https://wandb.ai/tomasposada67-universidad-eafit/amparo-legal-finetune/runs/d96zniak


train/entropy,█▅▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇███
train/global_step,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇███
train/grad_norm,█▃▂▁▁▁▁▂▁▂▂▂▂▂▂▂▂▃▂▂▃▃▂▂
train/learning_rate,██▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▁▁
train/loss,█▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/mean_token_accuracy,▁▆▇▇▇▇███████████████████
train/num_tokens,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇███
total_flos,4.422089958806016e+16
train/entropy,0.59476
train/epoch,3


## Guardar el adaptador LoRA en Drive

Se guarda directo en tu Drive (no solo son unos MB, son los pesos LoRA) para que no se pierda si el entorno de Colab se desconecta a mitad del entrenamiento o despues.

In [12]:
import shutil

model.save_pretrained(f'{OUTPUT_DIR}/adapter')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/adapter')

# ADAPTER_DIR viene de la celda de configuracion de Drive: una sola ruta,
# definida en un solo lugar, para que no se desincronice con el resto.
shutil.copytree(f'{OUTPUT_DIR}/adapter', ADAPTER_DIR, dirs_exist_ok=True)
print(f'Adaptador guardado en {ADAPTER_DIR}')

Adaptador guardado en /content/drive/MyDrive/Colab Notebooks/Amparo/amparo-lora-adapter


In [13]:
# prepare_model_for_kbit_training() dejo gradient checkpointing activo, lo que fuerza use_cache=False
# durante model.generate() (recalcula toda la atencion en cada token, sin KV-cache). Sin esto, el
# Paso 3 tarda varias veces mas de lo necesario y puede agotar la cuota de GPU de Colab a mitad de la evaluacion.
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

## Paso 3 -- Evaluar el modelo ya afinado (misma validacion)

In [14]:
finetuned_results = evaluate(model, val_records, label='fine-tuned')
guardar_resultados('finetuned_results.jsonl', finetuned_results)

[fine-tuned] 1/231  ruta=si  8.0s/ej  ETA ~30.5 min
[fine-tuned] 2/231  ruta=no   7.3s/ej  ETA ~27.8 min
[fine-tuned] 3/231  ruta=si  7.6s/ej  ETA ~28.9 min
[fine-tuned] 4/231  ruta=si  7.4s/ej  ETA ~28.0 min
[fine-tuned] 5/231  ruta=si  7.1s/ej  ETA ~26.8 min
[fine-tuned] 6/231  ruta=si  7.4s/ej  ETA ~27.6 min
[fine-tuned] 7/231  ruta=si  7.4s/ej  ETA ~27.6 min
[fine-tuned] 8/231  ruta=si  7.5s/ej  ETA ~27.8 min
[fine-tuned] 9/231  ruta=si  7.5s/ej  ETA ~27.7 min
[fine-tuned] 10/231  ruta=si  7.5s/ej  ETA ~27.6 min
[fine-tuned] 11/231  ruta=si  7.4s/ej  ETA ~27.3 min
[fine-tuned] 12/231  ruta=si  7.4s/ej  ETA ~27.1 min
[fine-tuned] 13/231  ruta=si  7.5s/ej  ETA ~27.1 min
[fine-tuned] 14/231  ruta=si  7.4s/ej  ETA ~26.8 min
[fine-tuned] 15/231  ruta=si  7.4s/ej  ETA ~26.8 min
[fine-tuned] 16/231  ruta=si  7.5s/ej  ETA ~26.8 min
[fine-tuned] 17/231  ruta=si  7.5s/ej  ETA ~26.8 min
[fine-tuned] 18/231  ruta=si  7.6s/ej  ETA ~26.8 min
[fine-tuned] 19/231  ruta=si  7.5s/ej  ETA ~26.6 min
[

## Paso 4 -- Comparacion baseline vs. afinado

Esta tabla es la evidencia real (no una suposicion) de si el fine-tuning mejoro las respuestas, por categoria y en promedio general.

**Como leerla.** La similitud lexica mide parecido con la respuesta de referencia, no correccion juridica: sube cuando el modelo adopta el formato y el registro del dataset. Con el dataset reescrito, lo que deberia aprender es a nombrar el mecanismo legal aplicable y la evidencia a reunir, asi que conviene mirar tambien algunas respuestas completas y no solo el numero.

**Esta cifra no es la del scorecard de M2** aunque midan lo mismo sobre la misma validacion: aqui se evalua el modelo en memoria y alla se recarga el adaptador desde Drive, normalmente en otra GPU (ver la nota de la seccion "Metrica de similitud"). La cifra de referencia del proyecto es la de M2, por estar respaldada por un artefacto comiteado con su manifiesto.

In [15]:
import pandas as pd

# Las banderas se recalculan aqui desde el texto generado, en vez de leer las
# que quedaron guardadas en cada registro. Si el catalogo de rutas se corrige
# despues de una corrida, esta tabla refleja la definicion vigente y no una
# medicion vieja: evita que el notebook muestre dos cifras distintas de lo mismo.
filas = []
for etiqueta, datos in [('baseline', baseline_results), ('fine_tuned', finetuned_results)]:
    for r in datos:
        filas.append({'id': r['id'], 'category': r['category'], 'modelo': etiqueta,
                      'nombra_mecanismo': nombra_mecanismo(r['generated']),
                      'cita_no_verificable': cita_no_verificable(r['generated']),
                      'palabras': len(r['generated'].split()),
                      'similitud_lexica': r['similarity']})
df = pd.DataFrame(filas)

# --- Resultado principal: las metricas de dominio --------------------------
resumen = df.groupby('modelo').agg(
    ruta_legal=('nombra_mecanismo', 'mean'),
    citas_no_verificables=('cita_no_verificable', 'mean'),
    palabras=('palabras', 'mean'),
    similitud_lexica=('similitud_lexica', 'mean'),
).reindex(['baseline', 'fine_tuned'])

print('=' * 72)
print('RESULTADO PRINCIPAL -- metricas de dominio (N =', len(baseline_results), 'de validacion)')
print('=' * 72)
print(f"{'metrica':<42}{'baseline':>12}{'fine-tuned':>14}")
print(f"{'Nombra una ruta legal concreta':<42}"
      f"{resumen.loc['baseline', 'ruta_legal']:>11.1%}{resumen.loc['fine_tuned', 'ruta_legal']:>14.1%}")
print(f"{'Cita normas que no puede verificar':<42}"
      f"{resumen.loc['baseline', 'citas_no_verificables']:>11.1%}"
      f"{resumen.loc['fine_tuned', 'citas_no_verificables']:>14.1%}")
print(f"{'Longitud media de la respuesta (palabras)':<42}"
      f"{resumen.loc['baseline', 'palabras']:>11.0f}{resumen.loc['fine_tuned', 'palabras']:>14.0f}")
print()
print(f"{'Similitud lexica (metrica de apoyo)':<42}"
      f"{resumen.loc['baseline', 'similitud_lexica']:>11.1f}"
      f"{resumen.loc['fine_tuned', 'similitud_lexica']:>14.1f}")
print('=' * 72)

# --- Desglose por categoria (ruta legal, que es lo que importa) -------------
por_cat = (df.pivot_table(index='category', columns='modelo', values='nombra_mecanismo', aggfunc='mean')
             .reindex(columns=['baseline', 'fine_tuned']))
por_cat['mejora'] = por_cat['fine_tuned'] - por_cat['baseline']
print('\nNombra una ruta legal, por categoria:')
print((por_cat * 100).round(0).astype('Int64').to_string())

df.to_csv(f'{SALIDA_DIR}/metricas_por_registro.csv', index=False)
por_cat.to_csv(f'{SALIDA_DIR}/metricas_por_categoria.csv')
print(f'\nCSV guardados en Drive: evaluacion/')

RESULTADO PRINCIPAL -- metricas de dominio (N = 231 de validacion)
metrica                                       baseline    fine-tuned
Nombra una ruta legal concreta                  96.1%         93.9%
Cita normas que no puede verificar              14.7%          0.0%
Longitud media de la respuesta (palabras)         229            46

Similitud lexica (metrica de apoyo)               3.2           6.0

Nombra una ruta legal, por categoria:
modelo                                           baseline  fine_tuned  mejora
category                                                                     
Acceso a informacion publica                          100         100       0
Accidentes de transito                                100         100       0
Arriendo                                              100          89     -11
Comparendos de transito                                89          78     -11
Conciliacion prejudicial                              100          75     -25
Contra

## Paso 4b -- Ejemplos reales de respuestas

Las cifras de arriba no se pueden juzgar sin ver que generó el modelo. Esta celda
imprime casos concretos (pregunta, respuesta de referencia, baseline y afinado) y
guarda una tabla en Drive para citarla en la documentación. Se muestran tanto
casos donde el fine-tuning mejoró como casos donde no, para no documentar solo lo
que conviene.

In [16]:
import pandas as pd

pares = list(zip(baseline_results, finetuned_results))
mejoro = [(b, f) for b, f in pares if f['nombra_mecanismo'] and not b['nombra_mecanismo']]
sin_cambio = [(b, f) for b, f in pares if not f['nombra_mecanismo']]

def mostrar(titulo, casos, n=3):
    print('=' * 78); print(titulo); print('=' * 78)
    for b, f in casos[:n]:
        print(f"\nCATEGORIA: {b['category']}")
        print(f"PREGUNTA:   {b['query'] if 'query' in b else b['messages'][1]['content']}")
        print(f"REFERENCIA: {b['messages'][2]['content']}")
        print(f"BASELINE:   {b['generated']}")
        print(f"AFINADO:    {f['generated']}")

mostrar(f'CASOS DONDE EL FINE-TUNING APORTO ({len(mejoro)} en total)', mejoro)
mostrar(f'CASOS DONDE EL AFINADO SIGUE SIN NOMBRAR UNA RUTA ({len(sin_cambio)} en total)', sin_cambio)

tabla = pd.DataFrame([{
    'id': b['id'], 'categoria': b['category'],
    'pregunta': b['messages'][1]['content'],
    'referencia': b['messages'][2]['content'],
    'baseline': b['generated'], 'afinado': f['generated'],
    'baseline_ruta': b['nombra_mecanismo'], 'afinado_ruta': f['nombra_mecanismo'],
} for b, f in pares])
tabla.to_csv(f'{SALIDA_DIR}/ejemplos_baseline_vs_afinado.csv', index=False)
print(f'\n\nTabla completa guardada en Drive: evaluacion/ejemplos_baseline_vs_afinado.csv')

CASOS DONDE EL FINE-TUNING APORTO (6 en total)

CATEGORIA: Peticiones de conducta ilegitima
PREGUNTA:   Puedo vender el carro que esta embargado sin decirle al comprador?
REFERENCIA: No: vender un bien con medidas vigentes y sin informarlo puede ser estafa, y el comprador podría demandarte. Pide primero en el juzgado el levantamiento del embargo, o informa la situación y acuerden el precio con eso a la vista. Pide el certificado de tradición del vehículo en el RUNT.
BASELINE:   No puedes vender un vehículo que está embargado sin informarle al comprador. El embargo judicial significa que el vehículo está bajo control del juez debido a una orden judicial. Venderlo sin el conocimiento del comprador y sin el consentimiento del juez sería fraudulento y puede ser considerado como un delito.

Para resolver esta situación, debes:

1. Informar al comprador sobre el embargo.
2. Buscar la autorización del juez para la venta.
3. Realizar la venta solo con la autorización judicial.

Si el comprador

## Paso 4c -- Publicar el adaptador en Hugging Face Hub

El adaptador no puede vivir solo en un Drive privado: sin acceso a ese Drive nadie
puede verificar los resultados ni reproducirlos. Subirlo al Hub lo vuelve
auditable y permite cargarlo desde cualquier parte con su identificador.

Necesitas un token de escritura de https://huggingface.co/settings/tokens.

In [17]:
from huggingface_hub import login, create_repo, upload_folder

HF_USER = 'Tomas0626'          # ajusta si tu usuario es otro
REPO_ID = f'{HF_USER}/amparo-lora-qwen2.5-7b'

login()   # pega tu token de escritura cuando lo pida

create_repo(REPO_ID, repo_type='model', exist_ok=True)
upload_folder(
    repo_id=REPO_ID,
    folder_path=ADAPTER_DIR,
    commit_message='Adaptador LoRA de M1 (dataset de 1410 ejemplos, 27 categorias)',
)
print(f'Adaptador publicado: https://huggingface.co/{REPO_ID}')

Adaptador publicado: https://huggingface.co/Tomas0626/amparo-lora-qwen2.5-7b
